In [5]:
import pyspark
from pyspark.sql import SparkSession

# Start Spark session

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [87]:
# Download homework dataset

!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-17 08:05:22--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.155.128.46, 18.155.128.6, 18.155.128.187, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.155.128.46|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.2’

yellow_tripdata_202 100%[===================>]  67.84M   192MB/s    in 0.4s    

2026-03-17 08:05:23 (192 MB/s) - ‘yellow_tripdata_2025-11.parquet.2’ saved [71134255/71134255]



In [9]:
# Read the downloaded file using pandas; save top 1000 rows as 'head_pd.parquet'.
# Verify the saved parquet file by opening it and counting rows.

import pandas as pd

# Take only first 1000 rows
df_pandas = pd.read_parquet('yellow_tripdata_2025-11.parquet', engine='pyarrow')
df_head = df_pandas.head(1000)

# Save as a valid parquet file
df_head.to_parquet('head_pd.parquet', index=False)

# Read the output file
df_pandas = pd.read_parquet('head_pd.parquet', engine='pyarrow')

# Verify length
print(len(df_pandas))

1000


In [10]:
# Read the downloaded file using pyarrow; save top 1000 rows as 'head_pd.parquet'.
# Verify the saved parquet file by opening it and counting rows.
# This method is faster cause it only reads first 1000 rows rather than wthe entire file.

import pyarrow.parquet as pq

# Read only first 1000 rows without loading the whole file
pf = pq.ParquetFile('yellow_tripdata_2025-11.parquet')
first_1000 = next(pf.iter_batches(batch_size=1000))
df_pandas = first_1000.to_pandas()

# Save as a valid parquet file
df_head.to_parquet('head_pq.parquet', index=False)

# Read the output file
df_pandas = pd.read_parquet('head_pq.parquet', engine='pyarrow')

# Verify length
print(len(df_pandas))

1000


In [11]:
# Read the data file with Spark; save down 1,000 rows, verify.
# This method is pretty fast too.

# Read
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Peek at the data
df.printSchema()
df.show(5)
print(f"Row count: {df.count()}")

# Take 1000 rows
df = df.limit(1000)

# Export to parquet
df.coalesce(1).write.parquet('output/', mode='overwrite')

# Read the output file
df_pandas = pd.read_parquet('output/', engine='pyarrow')

# Verify length
print(len(df_pandas))

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

[Stage 7:>                                                          (0 + 1) / 1]

1000


In [12]:
# Open Spark dataframe and check schema.

df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [13]:
# Schema looks ok, so move on to oartition the dataframe and save down.

df = df.repartition(24)
df.write.parquet('output/', mode='overwrite')

In [14]:
# Read repartitioned data file and check schema

df = spark.read.parquet('output/')
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [15]:
# Select a subset of columns and apply a filter

df.select('VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'total_amount') \
    .filter(df.trip_distance > 100) \
    .show(10)

+--------+--------------------+---------------------+------------+------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|total_amount|
+--------+--------------------+---------------------+------------+------------+------------+
|       2| 2025-11-09 20:26:57|  2025-11-10 02:13:51|         164|         265|      309.25|
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|         265|         265|       889.1|
|       2| 2025-11-20 21:27:00|  2025-11-20 21:54:00|          48|           4|        6.84|
|       2| 2025-11-20 07:29:00|  2025-11-20 07:48:00|         249|         162|       20.45|
|       2| 2025-11-17 05:35:00|  2025-11-17 05:55:00|         129|         234|       36.35|
|       2| 2025-11-12 13:28:00|  2025-11-12 14:01:00|          50|         152|        0.68|
|       2| 2025-11-19 14:11:00|  2025-11-19 14:48:00|         141|         231|       49.06|
|       2| 2025-11-07 07:50:00|  2025-11-07 08:14:00|          14|    

In [16]:
# Create new columns that pick up only the date component of datetime columns.

from pyspark.sql import functions as F, types

df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.tpep_dropoff_datetime)) \
    .select('VendorID', 'pickup_date', 'dropoff_date','PULocationID', 'DOLocationID', 'total_amount') \
    .show(10)

+--------+-----------+------------+------------+------------+------------+
|VendorID|pickup_date|dropoff_date|PULocationID|DOLocationID|total_amount|
+--------+-----------+------------+------------+------------+------------+
|       2| 2025-11-04|  2025-11-04|         132|         243|       95.88|
|       1| 2025-11-01|  2025-11-01|         144|         170|       18.95|
|       2| 2025-11-04|  2025-11-04|         143|         140|        22.3|
|       1| 2025-11-09|  2025-11-09|         180|         132|       105.6|
|       2| 2025-11-01|  2025-11-01|         236|         114|        38.7|
|       1| 2025-11-05|  2025-11-05|         100|         163|       16.23|
|       1| 2025-11-10|  2025-11-10|         170|         231|        23.6|
|       2| 2025-11-01|  2025-11-01|         114|         141|       28.62|
|       2| 2025-11-03|  2025-11-03|         238|         239|       12.74|
|       2| 2025-11-01|  2025-11-01|         132|         100|       99.78|
+--------+-----------+---

In [17]:
# Write a complex function that would be harder to write in SQL

def crazy_stuff(VendorID):
    num = VendorID + 1
    if num % 3 == 0:
        return f's/{num:03}'
    else:
        return f'e/{num:03}'

In [18]:
crazy_stuff(7)

'e/008'

In [19]:
# Make a user-defined function declaration and show similar slice of data, utilizing our user-defined fuction.

crazy_stuff_udf = F.udf(crazy_stuff, returnType = types.StringType())

df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.tpep_dropoff_datetime)) \
    .withColumn('ModVenID', crazy_stuff_udf(df.VendorID)) \
    .select('VendorID','ModVenID', 'pickup_date', 'dropoff_date','PULocationID', 'DOLocationID', 'total_amount') \
    .show(10)

[Stage 15:>                                                         (0 + 1) / 1]

+--------+--------+-----------+------------+------------+------------+------------+
|VendorID|ModVenID|pickup_date|dropoff_date|PULocationID|DOLocationID|total_amount|
+--------+--------+-----------+------------+------------+------------+------------+
|       2|   s/003| 2025-11-04|  2025-11-04|         132|         243|       95.88|
|       1|   e/002| 2025-11-01|  2025-11-01|         144|         170|       18.95|
|       2|   s/003| 2025-11-04|  2025-11-04|         143|         140|        22.3|
|       1|   e/002| 2025-11-09|  2025-11-09|         180|         132|       105.6|
|       2|   s/003| 2025-11-01|  2025-11-01|         236|         114|        38.7|
|       1|   e/002| 2025-11-05|  2025-11-05|         100|         163|       16.23|
|       1|   e/002| 2025-11-10|  2025-11-10|         170|         231|        23.6|
|       2|   s/003| 2025-11-01|  2025-11-01|         114|         141|       28.62|
|       2|   s/003| 2025-11-03|  2025-11-03|         238|         239|      